# Lesson 03 - Pola Desain Agenik

Dalam pelajaran ini, kita mengeksplorasi tiga pola desain dasar untuk membangun agen AI yang efektif:

1. **Instruksi Agen yang Jelas** — Membuat prompt yang tepat dan mendefinisikan peran guna mengarahkan perilaku agen  
2. **Output Terstruktur dengan Model Pydantic** — Memastikan agen mengembalikan data yang dapat diprediksi dan tervalidasi  
3. **Agen dengan Tanggung Jawab Tunggal** — Merancang agen fokus yang masing-masing melakukan satu tugas dengan baik

Kita akan menerapkan setiap pola pada skenario **rekomendasi destinasi perjalanan**, secara bertahap membangun sistem yang dapat menyarankan destinasi, memeriksa ketersediaan, dan menangani logistik.


## Pengaturan


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Pola 1: Instruksi Agen yang Jelas

Pola yang paling berdampak juga yang paling sederhana: menulis instruksi yang jelas dan rinci untuk agen Anda.

Instruksi yang baik mendefinisikan:
- **Siapa** agen itu (persona dan nada)
- **Apa** yang harus dilakukan (tanggung jawab langkah demi langkah)
- **Bagaimana** cara berperilaku (batasan dan gaya)

Di bawah ini, kami membuat agen konsier perjalanan dengan instruksi eksplisit yang membentuk setiap respons yang dihasilkannya.


In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Pola 2: Output Terstruktur dengan Model Pydantic

Teks bebas berguna untuk percakapan, tetapi sistem hilir memerlukan data terstruktur.  
Dengan menggabungkan **model Pydantic** dengan **fungsi alat**, kita dapat:

- Mendefinisikan skema yang tepat untuk output agen  
- Memvalidasi respons secara otomatis  
- Mengintegrasikan hasil agen ke dalam logika aplikasi dengan andal  

Kami juga memperkenalkan alat yang mengembalikan detail tujuan sehingga agen mendasarkan rekomendasinya pada data nyata.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = await provider.create_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## Pola 3: Agen Tanggung Jawab Tunggal

Tugas yang kompleks mendapat manfaat dari membagi pekerjaan di antara beberapa agen fokus, masing-masing dengan tanggung jawab tunggal:

- Seorang **Ahli Destinasi** yang mengetahui tentang tempat dan ketersediaan
- Seorang **Perencana Logistik** yang menangani penerbangan, hotel, dan jadwal perjalanan

Ini mencerminkan prinsip rekayasa perangkat lunak *pemisahan kepentingan* — setiap agen lebih mudah untuk diuji, dipelihara, dan ditingkatkan secara mandiri.


In [ ]:
destination_agent = await provider.create_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = await provider.create_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Ringkasan

Dalam pelajaran ini kami menerapkan tiga pola desain agen pada skenario rekomendasi perjalanan:

| Pola | Ide Utama | Manfaat |
|---|---|---|
| **Instruksi Jelas** | Tentukan persona, tanggung jawab, dan batasan di awal | Perilaku agen yang konsisten dan sesuai merek |
| **Output Terstruktur** | Gunakan model Pydantic sebagai format respons | Hasil yang tervalidasi dan dapat dibaca mesin |
| **Tanggung Jawab Tunggal** | Berikan setiap agen satu tugas fokus | Lebih mudah untuk diuji, dipelihara, dan dikomposisi |

Pola-pola ini dapat dikombinasikan secara alami — Anda dapat menggabungkan instruksi jelas dengan output terstruktur dalam agen bertanggung jawab tunggal untuk membangun sistem yang tangguh dan siap produksi.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Penafian**:  
Dokumen ini telah diterjemahkan menggunakan layanan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Meskipun kami berusaha untuk mencapai akurasi, harap diingat bahwa terjemahan otomatis mungkin mengandung kesalahan atau ketidakakuratan. Dokumen asli dalam bahasa aslinya harus dianggap sebagai sumber yang sah. Untuk informasi penting, disarankan menggunakan terjemahan oleh penerjemah profesional manusia. Kami tidak bertanggung jawab atas kesalahpahaman atau salah tafsir yang timbul dari penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
